# Paired CGR1 and reversible repair comparison

One proposed job: CGR1 control then RGR1 candidate,256updates each,512total,
seed1201,32steps. Shared600-second controlled cap covers BOTH arms,not10minutes
each. Setup/upload/export/idle extra. T4,Torch2.11.0+cu130,Python3.13.15,
NumPy2.1.3,CUDA13.0,cuDNN92700. Any mismatch stops. No retry or extra seeds.
Original TRAIN81 only; no curriculum. Models start fresh with identical parameters,
row order and firing RNG. Different update rule and supervised set: system-level
comparison,not a pure single-factor deletion ablation. Global CPU connectivity
cleanup remains enabled and included in candidate timing. A single timing probe
cannot guarantee the pair finishes under cap; partial evidence must be retained.

Open supplied notebook,upload matching ZIP. Keep APPROVED_SEED_JOB=False until
approval for this entire512-update job. After approval,set True and run once.
Download FULL ZIP+receipt,including failures. No Drive or automatic continuation.
Do not run the older compatibility or historical training scripts in the package.
Each arm retains per-update checkpoints,state,trace and boundary captures; RGR1
captures proposals,births,direct/cleanup removals,candidate and projected fields.
No quality-model claims follow from training completion.

Frozen review: both final256checkpoints only,CPUfloat32,32steps,firing2101,
original27development inputs, no TEST or threshold/horizon search. Report accepted
projected output plus pre-cleanup candidate diagnostics. Retain all original
acceptance gates: all9intact IoU>=.99 and valid; damaged validity>=17/18,
medianIoU>=.9705768039313023,excess<=325,recovered>=1945,median absolute volume
error<=19; zero original-input removals. Report regressions against same-runtime
control,repair deletions,cleanup deletions,oscillation,walltime and memory.
MG7 stays live. One seed does not establish generalization. No automatic admission.


In [ ]:
from google.colab import files
from pathlib import Path, PurePosixPath
import hashlib, zipfile, json, uuid, subprocess, sys
EXPECTED_SHA256 = '881f4fe54424dd0c9161cf6ee349589d0d53057875453b51223ab108e164a0cb'
ARCHIVE_NAME = 'NCA-RGR1-Paired-Package.zip'
uploaded = files.upload()
if len(uploaded) != 1: raise ValueError('Select exactly the supplied CGR3 ZIP')
raw = next(iter(uploaded.values()))
if hashlib.sha256(raw).hexdigest() != EXPECTED_SHA256: raise ValueError('Package checksum differs')
source = Path('/content') / ('cgr3-compat-upload-' + uuid.uuid4().hex + '.zip')
with source.open('xb') as f: f.write(raw)
PACKAGE = Path('/content') / ('nca-rgr1-pair-' + uuid.uuid4().hex)
with zipfile.ZipFile(source) as z:
    manifest = json.loads(z.read('manifest.json'))
    names = z.namelist()
    if len(names) != len(set(names)) or set(names) != set(manifest['files']) | {'manifest.json'}: raise ValueError('Unexpected members')
    if sum(x.file_size for x in z.infolist()) > 100_000_000: raise ValueError('Expanded size limit')
    for name in names:
        p = PurePosixPath(name)
        if p.is_absolute() or '..' in p.parts or '\\' in name or ':' in name or str(p) != name: raise ValueError('Unsafe path')
        if ((z.getinfo(name).external_attr >> 16) & 0o170000) == 0o120000: raise ValueError('Symlink member')
        if name != 'manifest.json' and hashlib.sha256(z.read(name)).hexdigest() != manifest['files'][name]: raise ValueError('Member hash differs')
    PACKAGE.mkdir()
    for name in names:
        target = PACKAGE / name
        target.parent.mkdir(parents=True, exist_ok=True)
        with target.open('xb') as f: f.write(z.read(name))
subprocess.run([sys.executable, '-c', 'from nca.reversible_package import verify; verify(".")'], cwd=PACKAGE, check=True)
print('Package verified:', PACKAGE)


In [ ]:
MODEL_SEED = 1201  # Only seed1201 is proposed.
APPROVED_SEED_JOB = False
if not APPROVED_SEED_JOB: raise RuntimeError('Stop until this seed job and backup plan are approved')
result = subprocess.run([sys.executable, str(PACKAGE / 'scripts/colab_reversible_pair.py'),
    '--seed', str(MODEL_SEED), '--device', 'cuda:0', '--approved-seed-job', '--seconds', '600'], cwd=PACKAGE)
print('Exit code:', result.returncode, '- download evidence next, including failures.')


In [ ]:
exports = sorted((PACKAGE / 'paired-runs').glob('*.zip'))
if not exports: raise RuntimeError('No export. Preserve runtime and send the error; do not rerun.')
for archive in exports:
    files.download(str(archive))
    files.download(str(archive.with_suffix('.receipt.json')))
print('Return ZIP and receipt for local verification; disconnect idle GPU after verified download.')
print('No additional seed or automatic retry is planned.')
